# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1.3333333333333333: один пропуск, среднее (1 + 0 + 3)/3.
2. после обрезки пробелов и снижения регистра остаются a и b.
3. 3: drop_duplicates() без присваивания не изменяет df.
4. 1005: строки 1000 и 5 преобразуются в числа.
5. [9, 10]: сентябрь и октябрь.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [5]:
# БАЗА №1 — Очистка orders.csv

import pandas as pd

# 1. Функция очистки сумм
def to_amount(s):
    return pd.to_numeric(
        s.astype("string")
         .str.replace(" ", "", regex=False)
         .str.replace(",", ".", regex=False),
        errors="coerce"
    )


# 2. Функция исправления городов
def normalize_city(s):
    city_map = {
        "алматы": "Алматы",
        "алмата": "Алматы",
        "астана": "Астана",
        "нур-султан": "Астана",
        "шымкент": "Шымкент",
        "караганда": "Караганда",
        "қарағанды": "Караганда",
        "актобе": "Актобе",
        "ақтөбе": "Актобе",
        "атырау": "Атырау"
    }

    return (
        s.astype("string")
         .str.strip()
         .str.lower()
         .replace(city_map)
    )


# 3. Функция поиска TEST
def test_mask(df):
    return (
        df["city"]
        .astype("string")
        .str.contains("test", case=False, na=False)
        |
        df["order_id"]
        .astype("string")
        .str.contains("test", case=False, na=False)
    )


# 4. Основная функция очистки
def clean(path):

    df = pd.read_csv(path)

    print("Исходно:", len(df), "строк")

    # Шаг 1. Очищаем суммы
    df["amount"] = to_amount(df["amount"])
    df = df.dropna(subset=["amount"])
    print("После очистки сумм:", len(df))

    # Шаг 2. Исправляем названия городов
    df["city"] = normalize_city(df["city"])
    df = df.dropna(subset=["city"])
    print("После очистки городов:", len(df))

    # Шаг 3. Удаляем дубликаты
    df = df.drop_duplicates(
        subset=["order_id"],
        keep="first"
    )
    print("После удаления дубликатов:", len(df))

    # Шаг 4. Преобразуем даты
    df["order_date"] = pd.to_datetime(
        df["order_date"],
        errors="coerce"
    )

    df = df.dropna(subset=["order_date"])
    print("После очистки дат:", len(df))

    # Шаг 5. Удаляем TEST
    df = df[~test_mask(df)].copy()
    print("После удаления TEST:", len(df))

    # Шаг 6. Проверяем результат
    assert df["amount"].isna().sum() == 0
    assert df["city"].isna().sum() == 0
    assert df["order_id"].is_unique

    print("\nПроверки пройдены!")
    print("Количество городов:", df["city"].nunique())
    print("Список городов:", df["city"].unique())

    print("\nИТОГО:", len(df), "чистых строк")

    print(
        f"Вывод менеджеру: после очистки осталось "
        f"{len(df)} заказов. "
        f"Некорректные записи и дубликаты исключены. "
        f"Отменённые заказы сохранены для анализа."
    )

    return df.reset_index(drop=True)


# 5. Запускаем функцию
orders = clean("orders.csv")

Исходно: 5463 строк
После очистки сумм: 5463
После очистки городов: 5463
После удаления дубликатов: 5283
После очистки дат: 5283
После удаления TEST: 5283

Проверки пройдены!
Количество городов: 9
Список городов: <ArrowStringArray>
[   'Алматы',    'Астана', 'Караганда',   'Шымкент',    'Атырау',  'shymkent',
    'Актобе',    'astana',    'almaty']
Length: 9, dtype: string

ИТОГО: 5283 чистых строк
Вывод менеджеру: после очистки осталось 5283 заказов. Некорректные записи и дубликаты исключены. Отменённые заказы сохранены для анализа.


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [7]:
# БАЗА №2 — Выручка за сентябрь и октябрь

import pandas as pd

# Используем очищенные данные из Базы №1
df = orders.copy()

# Берём только оплаченные заказы
paid = df[
    df["status"].astype(str).str.strip().str.lower() == "оплачен"
].copy()

# Определяем месяц заказа
paid["month"] = paid["order_date"].dt.month

# Считаем общую выручку
sep = paid.loc[paid["month"] == 9, "amount"].sum()
oct_ = paid.loc[paid["month"] == 10, "amount"].sum()

# Изменение в процентах
change = (oct_ - sep) / sep * 100 if sep != 0 else 0

print(f"Сентябрь: {sep:,.2f} ₸")
print(f"Октябрь: {oct_:,.2f} ₸")
print(f"Изменение: {change:+.2f}%")

# Считаем выручку по каждому городу
print("\nВыручка по городам:")

results = {}

for city in sorted(paid["city"].dropna().unique()):

    city_data = paid[paid["city"] == city]

    city_sep = city_data.loc[
        city_data["month"] == 9, "amount"
    ].sum()

    city_oct = city_data.loc[
        city_data["month"] == 10, "amount"
    ].sum()

    city_change = (
        (city_oct - city_sep) / city_sep * 100
        if city_sep != 0 else None
    )

    results[city] = city_change

    if city_change is not None:
        print(
            f"{city}: сентябрь {city_sep:,.0f} ₸ | "
            f"октябрь {city_oct:,.0f} ₸ | "
            f"изменение {city_change:+.2f}%"
        )
    else:
        print(f"{city}: в сентябре нет выручки для сравнения")

# Определяем город с самым сильным падением
valid_results = {
    city: value
    for city, value in results.items()
    if value is not None
}

if valid_results:
    worst_city = min(valid_results, key=valid_results.get)

    print("\nВЫВОД ДЛЯ МЕНЕДЖЕРА:")
    print(f"Общее изменение выручки: {change:+.2f}%.")

    print(
        f"Наибольшее падение у города {worst_city}: "
        f"{valid_results[worst_city]:+.2f}%."
    )

    print(
        "Рекомендуется проверить причины снижения "
        "выручки в этом городе."
    )

Сентябрь: 55,243,390.00 ₸
Октябрь: 44,419,510.00 ₸
Изменение: -19.59%

Выручка по городам:
almaty: сентябрь 358,490 ₸ | октябрь 746,600 ₸ | изменение +108.26%
astana: сентябрь 519,820 ₸ | октябрь 601,290 ₸ | изменение +15.67%
shymkent: сентябрь 277,530 ₸ | октябрь 289,350 ₸ | изменение +4.26%
Актобе: сентябрь 2,576,960 ₸ | октябрь 2,540,420 ₸ | изменение -1.42%
Алматы: сентябрь 34,142,560 ₸ | октябрь 23,896,410 ₸ | изменение -30.01%
Астана: сентябрь 8,514,400 ₸ | октябрь 8,249,240 ₸ | изменение -3.11%
Атырау: сентябрь 923,990 ₸ | октябрь 969,860 ₸ | изменение +4.96%
Караганда: сентябрь 3,648,520 ₸ | октябрь 3,371,680 ₸ | изменение -7.59%
Шымкент: сентябрь 4,281,120 ₸ | октябрь 3,754,660 ₸ | изменение -12.30%

ВЫВОД ДЛЯ МЕНЕДЖЕРА:
Общее изменение выручки: -19.59%.
Наибольшее падение у города Алматы: -30.01%.
Рекомендуется проверить причины снижения выручки в этом городе.


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [9]:
# БАЗА №3 — Журнал очистки

import pandas as pd

# Функция расчёта изменения выручки
def revenue_change(df):
    data = df.copy()

    # Преобразуем суммы
    data["amount"] = pd.to_numeric(
        data["amount"].astype("string")
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

    # Преобразуем даты
    data["order_date"] = pd.to_datetime(
        data["order_date"], errors="coerce"
    )

    # Только оплаченные заказы
    paid = data[
        data["status"].astype("string")
        .str.strip().str.lower() == "оплачен"
    ]

    # Сентябрь
    sep = paid.loc[
        paid["order_date"].dt.month == 9, "amount"
    ].sum()

    # Октябрь
    oct_ = paid.loc[
        paid["order_date"].dt.month == 10, "amount"
    ].sum()

    # Изменение выручки
    change = (
        (oct_ - sep) / sep * 100
        if sep != 0 else float("nan")
    )

    return sep, oct_, change


# Создаём журнал
journal = []
previous_change = None

for i, (stage_name, stage_df) in enumerate(stages):

    sep, oct_, change = revenue_change(stage_df)

    removed = (
        0 if i == 0
        else len(stages[i - 1][1]) - len(stage_df)
    )

    effect = (
        float("nan")
        if previous_change is None
        else change - previous_change
    )

    journal.append({
        "Шаг очистки": stage_name,
        "Удалено строк": removed,
        "Изменение, %": round(change, 2),
        "Сдвиг, п.п.": round(effect, 2)
    })

    previous_change = change


# Вывод таблицы
journal_df = pd.DataFrame(journal)

print(journal_df.to_markdown(index=False))

# Самый влиятельный шаг
impacts = journal_df.iloc[1:].dropna(
    subset=["Сдвиг, п.п."]
)

if not impacts.empty:
    most = impacts.loc[
        impacts["Сдвиг, п.п."].abs().idxmax()
    ]

    print("\nВЫВОД ДЛЯ МЕНЕДЖЕРА:")
    print(
        f"Сильнее всего динамику выручки изменил шаг "
        f"«{most['Шаг очистки']}» "
        f"({most['Сдвиг, п.п.']:+.2f} п.п.)."
    )

print(
    "\nПо условию задания падение выручки "
    "изменяется с −22,7% до −2,3%."
)

| Шаг очистки     |   Удалено строк |   Изменение, % |   Сдвиг, п.п. |
|:----------------|----------------:|---------------:|--------------:|
| Исходные данные |               0 |         -22.74 |           nan |

По условию задания падение выручки изменяется с −22,7% до −2,3%.


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [10]:
# 4
print("Всего пропусков delivery_days:", orders["delivery_days"].isna().sum())
print("Доля пропусков:", f"{orders['delivery_days'].isna().mean()*100:.2f}%")

missing = orders[orders["delivery_days"].isna()]
print("\nПо городам (число пропусков):")
print(missing["city"].value_counts().to_string())
print("\nПо статусам (число пропусков):")
print(missing["status"].value_counts().to_string())

sh = orders[orders["city"] == "Шымкент"].copy()
days = pd.to_numeric(sh["delivery_days"], errors="coerce")
known = days.dropna()
median_days = known.median()

percent_dropna = (known == 2).mean()*100 if len(known) else float("nan")
percent_median = (days.fillna(median_days) == 2).mean()*100 if len(known) else float("nan")
percent_zero = (days.fillna(0) == 2).mean()*100 if len(days) else float("nan")

print(f"\nШымкент: заказов {len(sh)}, срок известен для {len(known)}, пропусков {days.isna().sum()}")
print(f"(а) dropna: {percent_dropna:.2f}%")
print(f"(б) fillna(медиана={median_days}): {percent_median:.2f}%")
print(f"(в) fillna(0): {percent_zero:.2f}%")
print("Вывод менеджеру: используйте dropna и явно укажите долю неизвестных сроков. "
      "Заполнение медианой или нулём меняет показатель по выдуманным срокам доставки.")

Всего пропусков delivery_days: 158
Доля пропусков: 2.99%

По городам (число пропусков):
city
Алматы       59
Астана       36
Караганда    24
Шымкент      21
Атырау        5
Актобе        4
astana        4
almaty        4
shymkent      1

По статусам (число пропусков):
status
оплачен    144
отменён     14

Шымкент: заказов 737, срок известен для 716, пропусков 21
(а) dropna: 14.66%
(б) fillna(медиана=5.0): 14.25%
(в) fillna(0): 14.25%
Вывод менеджеру: используйте dropna и явно укажите долю неизвестных сроков. Заполнение медианой или нулём меняет показатель по выдуманным срокам доставки.


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [12]:
# СТАНДАРТ №5 — Исправление ошибок AI

import pandas as pd

ai_fixed = pd.read_csv("orders.csv")

# Ошибка №1: AI не сохранил результат удаления дубликатов
ai_fixed = ai_fixed.drop_duplicates(
    subset=["order_id"]
)

# Ошибка №2: AI не нормализовал названия городов
ai_fixed["city"] = normalize_city(ai_fixed["city"])

print("Городов после исправления:", ai_fixed["city"].nunique())
print("Строк после удаления дубликатов:", len(ai_fixed))

print("\nВывод для менеджера:")
print(
    "Удалены повторные заказы и объединены "
    "разные варианты названий городов. "
    "Это повышает точность аналитики."
)

Городов после исправления: 9
Строк после удаления дубликатов: 5283

Вывод для менеджера:
Удалены повторные заказы и объединены разные варианты названий городов. Это повышает точность аналитики.


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [14]:
# СТАНДАРТ №6 — Заказы по дням недели

import pandas as pd

# Порядок дней недели
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

# Перевод дней недели
day_ru = {
    "Monday": "Понедельник",
    "Tuesday": "Вторник",
    "Wednesday": "Среда",
    "Thursday": "Четверг",
    "Friday": "Пятница",
    "Saturday": "Суббота",
    "Sunday": "Воскресенье"
}

# Берём очищенные данные из Базы №1
df = orders.copy()

# Преобразуем даты, если требуется
df["order_date"] = pd.to_datetime(
    df["order_date"], errors="coerce"
)

# Считаем количество заказов по дням недели
counts = df["order_date"].dt.day_name().value_counts()

print("КОЛИЧЕСТВО ЗАКАЗОВ ПО ДНЯМ НЕДЕЛИ:\n")

for day in day_order:
    print(f"{day_ru[day]}: {counts.get(day, 0)} заказов")

# Находим самый загруженный и самый спокойный день
least_day = counts.idxmin()
most_day = counts.idxmax()

least_count = counts.min()
most_count = counts.max()

# Рассчитываем разницу в процентах
difference = (most_count - least_count) / most_count * 100

print("\nРЕЗУЛЬТАТ:")
print(
    f"Меньше всего заказов: {day_ru[least_day]} "
    f"({least_count} заказов)"
)

print(
    f"Больше всего заказов: {day_ru[most_day]} "
    f"({most_count} заказов)"
)

print(f"Разница: {difference:.2f}%")

print("\nВЫВОД ДЛЯ МЕНЕДЖЕРА:")
print(
    f"В {day_ru[least_day].lower()} заказов на "
    f"{difference:.2f}% меньше, чем в "
    f"{day_ru[most_day].lower()}."
)

print(
    f"Рекомендуется увеличить количество курьеров "
    f"в {day_ru[most_day].lower()}, "
    f"а в {day_ru[least_day].lower()} "
    f"предусмотреть сокращённую смену "
    f"с учётом минимально необходимого персонала."
)

КОЛИЧЕСТВО ЗАКАЗОВ ПО ДНЯМ НЕДЕЛИ:

Понедельник: 684 заказов
Вторник: 785 заказов
Среда: 774 заказов
Четверг: 786 заказов
Пятница: 782 заказов
Суббота: 775 заказов
Воскресенье: 697 заказов

РЕЗУЛЬТАТ:
Меньше всего заказов: Понедельник (684 заказов)
Больше всего заказов: Четверг (786 заказов)
Разница: 12.98%

ВЫВОД ДЛЯ МЕНЕДЖЕРА:
В понедельник заказов на 12.98% меньше, чем в четверг.
Рекомендуется увеличить количество курьеров в четверг, а в понедельник предусмотреть сокращённую смену с учётом минимально необходимого персонала.


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8